# Tutorial 1: Simulation Exploration

This tutorial demonstrates how to use the Ravan Quantum-ML System's simulation modules.

## Overview

The Ravan system includes three physics simulators:
1. **Quantum Circuit Simulator** - Simulates quantum circuits with entanglement
2. **Schrödinger Solver** - Solves 1D time-dependent Schrödinger equation
3. **Harmonic Oscillator** - Simulates quantum harmonic oscillator dynamics

Let's explore each one!

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from quantum_circuit_simulator import QuantumCircuitSimulator
from schrodinger_solver import SchrodingerSolver
from harmonic_oscillator import HarmonicOscillator

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Quantum Circuit Simulator

Let's create a Bell state and analyze its entanglement properties.

In [ ]:
# Initialize simulator
qc_sim = QuantumCircuitSimulator()

# Run simulation with Bell state
params = {
    'n_qubits': 2,
    'shots': 1000,
    'gate_sequence': 'bell'
}

result = qc_sim.run(params)

print("Quantum Circuit Results:")
print(f"  Entropy: {result['entropy']:.4f}")
print(f"  Chi-squared: {result['chi_squared']:.4f}")
print(f"  KL Divergence: {result['kl_divergence']:.4f}")
print(f"\nMeasurement counts: {result['counts']}")

In [ ]:
# Visualize measurement results
counts = result['counts']
states = list(counts.keys())
probabilities = [counts[s] / params['shots'] for s in states]

plt.figure(figsize=(10, 6))
plt.bar(states, probabilities, color='steelblue', alpha=0.7)
plt.xlabel('Quantum State', fontsize=12)
plt.ylabel('Probability', fontsize=12)
plt.title('Bell State Measurement Distribution', fontsize=14)
plt.ylim(0, 0.6)
plt.grid(axis='y', alpha=0.3)
plt.show()

print("\nNote: Bell states show equal probability for |00⟩ and |11⟩, indicating maximal entanglement!")

## 2. Schrödinger Equation Solver

Let's simulate quantum tunneling through a potential barrier.

In [ ]:
# Initialize solver
schrodinger = SchrodingerSolver()

# Run simulation with barrier
params = {
    'V0': 5.0,           # Barrier height
    'barrier_width': 1.5, # Barrier width
    'k0': 4.0,           # Initial momentum
    'sigma': 1.0,        # Wavepacket width
    'x0': -5.0           # Initial position
}

result = schrodinger.run(params)

print("Schrödinger Solver Results:")
print(f"  Transmission coefficient: {result['transmission']:.4f}")
print(f"  Reflection coefficient: {result['reflection']:.4f}")
print(f"  Total probability: {result['total_probability']:.4f}")
print(f"  Energy: {result['energy']:.4f}")
print(f"\n  Conservation check (T+R): {result['transmission'] + result['reflection']:.4f}")

In [ ]:
# Explore tunneling vs barrier height
barrier_heights = np.linspace(1.0, 10.0, 20)
transmissions = []

for V0 in barrier_heights:
    params['V0'] = V0
    result = schrodinger.run(params)
    transmissions.append(result['transmission'])

plt.figure(figsize=(10, 6))
plt.plot(barrier_heights, transmissions, 'o-', color='darkred', linewidth=2, markersize=6)
plt.xlabel('Barrier Height (V₀)', fontsize=12)
plt.ylabel('Transmission Coefficient', fontsize=12)
plt.title('Quantum Tunneling: Transmission vs Barrier Height', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

print("\nObservation: Higher barriers lead to exponentially decreasing transmission!")

## 3. Harmonic Oscillator

Let's simulate a quantum harmonic oscillator and verify energy level spacing.

In [ ]:
# Initialize oscillator
oscillator = HarmonicOscillator()

# Run simulation
params = {
    'oscillator_length': 1.0,
    'basis_size': 10,
    'initial_n': 2
}

result = oscillator.run(params)

print("Harmonic Oscillator Results:")
print(f"  Energy levels: {result['energy_levels'][:5]}")
print(f"  Expected photon number: {result['expected_n']:.4f}")
print(f"  Coherence: {result['coherence']:.4f}")

In [ ]:
# Visualize energy levels
energy_levels = result['energy_levels']
n_levels = len(energy_levels)

plt.figure(figsize=(10, 6))
plt.plot(range(n_levels), energy_levels, 'o-', color='darkgreen', linewidth=2, markersize=8)
plt.xlabel('Quantum Number (n)', fontsize=12)
plt.ylabel('Energy (ℏω)', fontsize=12)
plt.title('Quantum Harmonic Oscillator Energy Levels', fontsize=14)
plt.grid(alpha=0.3)
plt.show()

# Check uniform spacing
spacings = np.diff(energy_levels)
print(f"\nEnergy level spacings: {spacings[:5]}")
print(f"Average spacing: {np.mean(spacings):.4f} (should be ~1.0)")
print(f"Spacing variance: {np.var(spacings):.6f} (should be ~0)")

## 4. Parameter Space Exploration

Let's explore how different parameters affect the observables.

In [ ]:
# Explore Schrödinger solver parameter space
k0_values = np.linspace(2.0, 8.0, 15)
V0_values = [3.0, 5.0, 7.0]

plt.figure(figsize=(12, 6))

for V0 in V0_values:
    transmissions = []
    for k0 in k0_values:
        params = {
            'V0': V0,
            'barrier_width': 1.5,
            'k0': k0,
            'sigma': 1.0,
            'x0': -5.0
        }
        result = schrodinger.run(params)
        transmissions.append(result['transmission'])
    
    plt.plot(k0_values, transmissions, 'o-', label=f'V₀ = {V0}', linewidth=2, markersize=5)

plt.xlabel('Initial Momentum (k₀)', fontsize=12)
plt.ylabel('Transmission Coefficient', fontsize=12)
plt.title('Transmission vs Momentum for Different Barrier Heights', fontsize=14)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.show()

print("\nKey insight: Higher momentum increases tunneling probability!")

## 5. Physics Validation

All simulators include built-in physics validation. Let's verify conservation laws.

In [ ]:
# Test multiple simulations and check conservation
n_tests = 50
conservation_errors = []

for i in range(n_tests):
    # Random parameters
    params = {
        'V0': np.random.uniform(2.0, 8.0),
        'barrier_width': np.random.uniform(1.0, 2.0),
        'k0': np.random.uniform(3.0, 6.0),
        'sigma': 1.0,
        'x0': -5.0
    }
    
    result = schrodinger.run(params)
    conservation_error = abs(result['transmission'] + result['reflection'] - 1.0)
    conservation_errors.append(conservation_error)

plt.figure(figsize=(10, 6))
plt.hist(conservation_errors, bins=20, color='purple', alpha=0.7, edgecolor='black')
plt.xlabel('Conservation Error |T + R - 1|', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Probability Conservation Validation (50 Random Simulations)', fontsize=14)
plt.axvline(0.001, color='red', linestyle='--', linewidth=2, label='Tolerance (0.001)')
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f"\nMax conservation error: {max(conservation_errors):.6f}")
print(f"Mean conservation error: {np.mean(conservation_errors):.6f}")
print(f"All tests pass: {all(e < 0.001 for e in conservation_errors)}")

## Summary

In this tutorial, you learned:

1. ✅ How to use all three simulation modules
2. ✅ How to interpret physical observables
3. ✅ How to explore parameter spaces
4. ✅ How physics validation ensures correctness

**Next Steps:**
- Tutorial 2: Train ML models on simulation data
- Tutorial 3: Analyze and interpret model predictions

**Try it yourself:**
- Modify parameters and observe changes
- Create custom parameter sweeps
- Combine multiple simulators in one workflow